# 🚨 AOG 자재 수급 어시스턴트

좌측 AI 상황 요약 + 우측 6단계 프로세스 진행을 한 화면에서 보여주는 Streamlit 앱입니다. 아래 3개 셀을 순서대로 실행하세요. 사용법·테스트 시나리오는 저장소의 `README.md`를 참고하세요.

## Cell 1 — 라이브러리 설치 (설치 후 런타임 자동 재시작됨, 정상 동작)

Colab은 numpy/pandas/torch가 이미 메모리에 로드된 상태로 시작합니다. 이 상태에서 새 패키지를 설치하면 버전이 어긋나며 위젯 전반에서 TypeError가 나는 것이 Colab의 잘 알려진 증상이라, 설치 직후 런타임을 자동으로 재시작합니다. **"세션이 다운되었습니다" 경고가 떠도 정상이며, 무시하고 Cell 2부터 다시 실행하면 됩니다.**

의존성 충돌을 줄이기 위해 다음 원칙을 따릅니다.
- `numpy` 버전을 강제로 고정하지 않습니다 — Colab에 이미 설치된 torch/pandas와 어긋날 수 있는 인위적인 제약이라, 재시작만으로 충분한 앞의 원인을 오히려 재현시킬 수 있습니다.
- 앱이 실제로 필요로 하는 필수 패키지(`streamlit`, `plotly`)와 선택 패키지(`transformers`)의 설치 명령을 분리했습니다 — `transformers` 설치가 느리거나 실패해도 필수 기능은 영향받지 않습니다.
- 이미 만족하는 의존성까지 전부 끌어올리지 않도록 `--upgrade`는 꼭 필요한 패키지에만 사용합니다.
- 실제로 쓰지 않는 `accelerate`는 설치 목록에서 뺐습니다(코드가 `device_map`을 쓰지 않음).

`transformers`는 좌측 패널의 선택적 로컬 LLM 브리핑/메일 초안 기능에 쓰입니다 — 설치가 실패하거나 느려도 규칙 기반 요약·기본 메일 템플릿으로 항상 정상 동작합니다.

In [ ]:
!pip install -q -U streamlit plotly
!pip install -q transformers

import os
os.kill(os.getpid(), 9)  # 방금 설치한 패키지가 깨끗하게 적용되도록 런타임 자동 재시작


## Cell 2 — Streamlit 앱 코드 작성 (`app.py`)

In [ ]:
%%writefile app.py
# ============================================================================
# AOG(Aircraft on Ground) 자재 수급 어시스턴트 — Streamlit
# ----------------------------------------------------------------------------
# 좌측: AI(로컬 LLM, 기본 자동 실행) 상황 요약 & 추천 행동
# 우측: FAK -> Allocation -> Pooling -> Main Station 타사 -> 동일기종 타사 -> Hand-carry
#       6단계 프로세스를 사용자가 버튼으로 직접 확인하며 진행 (자동 진행 없음)
# 사이드바: "AOG 대시보드" / "실시간 운항 현황" / "데이터 관리" 세 화면 전환
#          (참조 데이터는 JSON 파일로 영속화, 항공편 정보는 API 연동 포인트를 둔 더미 공급자에서 조회)
# ============================================================================

import copy
import json
import os
import random
import re
import urllib.parse
from datetime import datetime, timedelta

import pandas as pd
import plotly.graph_objects as go
import streamlit as st

try:
    import transformers  # noqa: F401
    HAS_TRANSFORMERS = True
except ImportError:
    HAS_TRANSFORMERS = False

st.set_page_config(page_title="AOG 자재 수급 어시스턴트", page_icon="🚨", layout="wide")

DB_PATH = "aog_db.json"

# ----------------------------------------------------------------------------
# 1. 더미 데이터 (기본값) — 파일이 없을 때만 이 값으로 aog_db.json이 생성된다.
#    실제 운영 시에는 관리자가 "데이터 관리" 화면에서 언제든 갱신할 수 있다.
#
#    주의: "기종별 대여 가능 재고"는 의도적으로 포함하지 않는다. 타 항공사가 특정
#    자재를 실제 대여 가능한 수량으로 보유하고 있는지는 그 항공사의 기밀 정보라
#    사전에 알 수 없다 — 그래서 5단계는 항상 "문의가 필요한 상태"로 시작하고,
#    "기종별 운영 항공사" 목록만으로 문의 대상을 좁혀준다.
# ----------------------------------------------------------------------------

DEFAULT_DB = {
    "fak_stock": [
        {"aircraft_type": "A330-300", "part_number": "OXY-GEN-A330-15", "qty": 2, "location": "ICN FAK 창고 A-12"},
        {"aircraft_type": "B737-800", "part_number": "SMK-DET-737-04", "qty": 3, "location": "ICN FAK 창고 B-05"},
        {"aircraft_type": "B787-9", "part_number": "EVAC-SLIDE-787-02", "qty": 1, "location": "ICN FAK 창고 C-01"},
        {"aircraft_type": "A321neo", "part_number": "CAB-PRESS-VLV-321N-06", "qty": 4, "location": "ICN FAK 창고 A-09"},
        {"aircraft_type": "A350-900", "part_number": "SMOKE-DET-350-11", "qty": 2, "location": "ICN FAK 창고 D-03"},
        {"aircraft_type": "B777-300ER", "part_number": "TIRE-MLG-777-19", "qty": 2, "location": "ICN FAK 창고 B-14"},
    ],
    "allocation_stock": [
        {"aircraft_type": "B777-300ER", "part_number": "BRK-B777-CARBON-01", "qty": 1, "location": "ICN 본사 Allocation 창고"},
        {"aircraft_type": "A330-300", "part_number": "FUEL-NOZ-A330-07", "qty": 4, "location": "ICN 본사 Allocation 창고"},
        {"aircraft_type": "B737-800", "part_number": "STARTER-GEN-737-03", "qty": 2, "location": "ICN 본사 Allocation 창고"},
        {"aircraft_type": "A321neo", "part_number": "FLAP-TRK-ROLLER-321N-08", "qty": 5, "location": "ICN 본사 Allocation 창고"},
        {"aircraft_type": "B787-9", "part_number": "APU-787-01", "qty": 1, "location": "ICN 본사 Allocation 창고"},
        {"aircraft_type": "A350-900", "part_number": "IDG-A350-002", "qty": 1, "location": "ICN 본사 Allocation 창고"},
    ],
    "pooling_partners": [
        {"partner": "SIA Engineering (싱가포르)", "contact": "+65-6541-2000", "email": "poolingdesk@siae.com.sg"},
        {"partner": "Lufthansa Technik (독일)", "contact": "+49-40-5070-5553", "email": "pooling@lht.dlh.de"},
        {"partner": "ANA Base Maintenance (일본)", "contact": "+81-3-6735-1111", "email": "pooling@ana.co.jp"},
        {"partner": "HAECO (홍콩)", "contact": "+852-2767-6111", "email": "pooling@haeco.com"},
        {"partner": "AAR Corp (미국)", "contact": "+1-630-227-2000", "email": "pooling@aarcorp.com"},
    ],
    "pooling_stock": [
        {"partner": "SIA Engineering (싱가포르)", "aircraft_type": "B737-800", "part_number": "HYD-PUMP-737-11", "qty": 1, "location": "SIN 창고"},
        {"partner": "Lufthansa Technik (독일)", "aircraft_type": "A330-300", "part_number": "IDG-A330-001", "qty": 1, "location": "FRA 창고"},
        {"partner": "HAECO (홍콩)", "aircraft_type": "B777-300ER", "part_number": "WHL-MLG-777-06", "qty": 2, "location": "HKG 창고"},
        {"partner": "AAR Corp (미국)", "aircraft_type": "A321neo", "part_number": "AVIONICS-FMS-321N-04", "qty": 1, "location": "ORD 창고"},
        {"partner": "ANA Base Maintenance (일본)", "aircraft_type": "B787-9", "part_number": "CARGO-DOOR-ACT-787-03", "qty": 1, "location": "NRT 창고"},
    ],
    "main_station_airlines": [
        {"airport": "ICN", "airline": "아시아나항공", "contact": "02-2669-8000", "email": "ops.icn@flyasiana.com"},
        {"airport": "GMP", "airline": "제주항공", "contact": "02-2015-1000", "email": "ops.gmp@jejuair.net"},
        {"airport": "NRT", "airline": "ANA", "contact": "+81-3-6735-1000", "email": "ops.nrt@ana.co.jp"},
        {"airport": "HND", "airline": "JAL", "contact": "+81-3-5460-3121", "email": "ops.hnd@jal.com"},
        {"airport": "CDG", "airline": "Air France", "contact": "+33-1-4356-7890", "email": "ops.cdg@airfrance.fr"},
        {"airport": "JFK", "airline": "Delta Air Lines", "contact": "+1-800-221-1212", "email": "ops.jfk@delta.com"},
        {"airport": "SIN", "airline": "Singapore Airlines", "contact": "+65-6223-8888", "email": "ops.sin@singaporeair.com"},
        {"airport": "HKG", "airline": "Cathay Pacific", "contact": "+852-2747-1888", "email": "ops.hkg@cathaypacific.com"},
        {"airport": "FRA", "airline": "Lufthansa", "contact": "+49-69-86799799", "email": "ops.fra@lufthansa.com"},
        {"airport": "LAX", "airline": "United Airlines", "contact": "+1-800-864-8331", "email": "ops.lax@united.com"},
    ],
    "other_airline_station_stock": [
        {"airport": "GMP", "part_number": "CAB-DOOR-ACT-321N-03", "airline": "제주항공", "qty": 1,
         "location": "GMP", "contact": "02-2015-1000", "email": "ops.gmp@jejuair.net"},
        {"airport": "HKG", "part_number": "WHL-MLG-777-06", "airline": "Cathay Pacific", "qty": 1,
         "location": "HKG", "contact": "+852-2747-1888", "email": "ops.hkg@cathaypacific.com"},
    ],
    # 기종별 "운영 항공사"만 관리한다. 실 재고 보유 여부는 항공사 기밀 정보라 사전 등록하지 않으며,
    # 5단계는 이 목록으로 문의 대상만 좁혀서 매번 실제 문의(메일)를 보내도록 설계되어 있다.
    "fleet_operators": [
        {"aircraft_type": "A330-300", "airline": "아시아나항공", "contact": "02-2669-8000", "email": "fleet.a330@flyasiana.com"},
        {"aircraft_type": "A330-300", "airline": "Cathay Pacific", "contact": "+852-2747-1888", "email": "fleet.a330@cathaypacific.com"},
        {"aircraft_type": "A330-300", "airline": "China Eastern", "contact": "+86-21-95530", "email": "fleet.a330@ceair.com"},
        {"aircraft_type": "B777-300ER", "airline": "아시아나항공", "contact": "02-2669-8000", "email": "fleet.b777@flyasiana.com"},
        {"aircraft_type": "B777-300ER", "airline": "Emirates", "contact": "+971-600-555555", "email": "fleet.b777@emirates.com"},
        {"aircraft_type": "B777-300ER", "airline": "ANA", "contact": "+81-3-6735-1000", "email": "fleet.b777@ana.co.jp"},
        {"aircraft_type": "B737-800", "airline": "제주항공", "contact": "02-2015-1000", "email": "fleet.b737@jejuair.net"},
        {"aircraft_type": "B737-800", "airline": "티웨이항공", "contact": "1688-8686", "email": "fleet.b737@twayair.com"},
        {"aircraft_type": "B737-800", "airline": "Ryanair", "contact": "+353-1-945-1212", "email": "fleet.b737@ryanair.com"},
        {"aircraft_type": "A321neo", "airline": "에어부산", "contact": "1666-3060", "email": "fleet.a321n@airbusan.com"},
        {"aircraft_type": "A321neo", "airline": "IndiGo", "contact": "+91-124-6173838", "email": "fleet.a321n@goindigo.in"},
        {"aircraft_type": "A321neo", "airline": "Wizz Air", "contact": "+36-1-777-9300", "email": "fleet.a321n@wizzair.com"},
        {"aircraft_type": "B787-9", "airline": "ANA", "contact": "+81-3-6735-1000", "email": "fleet.b787@ana.co.jp"},
        {"aircraft_type": "B787-9", "airline": "United Airlines", "contact": "+1-800-864-8331", "email": "fleet.b787@united.com"},
        {"aircraft_type": "B787-9", "airline": "Etihad Airways", "contact": "+971-2-599-0000", "email": "fleet.b787@etihad.ae"},
        {"aircraft_type": "A350-900", "airline": "아시아나항공", "contact": "02-2669-8000", "email": "fleet.a350@flyasiana.com"},
        {"aircraft_type": "A350-900", "airline": "Cathay Pacific", "contact": "+852-2747-1888", "email": "fleet.a350@cathaypacific.com"},
        {"aircraft_type": "A350-900", "airline": "Qatar Airways", "contact": "+974-4023-0000", "email": "fleet.a350@qatarairways.com.qa"},
    ],
    "allocation_dept_contacts": [
        {"airport": "ICN", "department": "자재관리팀 Allocation 파트", "contact": "02-XXXX-1000", "email": "allocation.icn@airline.example"},
        {"airport": "GMP", "department": "자재관리팀 Allocation 파트(김포)", "contact": "02-XXXX-1005", "email": "allocation.gmp@airline.example"},
        {"airport": "NRT", "department": "해외지점 Allocation 파트(나리타)", "contact": "+81-3-XXXX-1010", "email": "allocation.nrt@airline.example"},
        {"airport": "CDG", "department": "해외지점 Allocation 파트(파리)", "contact": "+33-1-XXXX-1015", "email": "allocation.cdg@airline.example"},
        {"airport": "HKG", "department": "해외지점 Allocation 파트(홍콩)", "contact": "+852-XXXX-1020", "email": "allocation.hkg@airline.example"},
    ],
    "customs_team": {"name": "통관팀", "contact": "02-XXXX-2000", "email": "customs@airline.example"},
    # flight_schedule은 더 이상 여기서 수동 관리하지 않는다 — 실시간 항공편 API(현재는 더미 공급자)에서
    # 조회한다. 아래 "4.5 실시간 항공편/공항 데이터 공급자" 섹션 참고.
}

# 자재 넘버 드롭다운(자유 입력이지만 편의상 후보도 제공)에 노출할 카탈로그 — README 시나리오와 1:1 대응
PART_NUMBER_CATALOG = [
    "OXY-GEN-A330-15", "BRK-B777-CARBON-01", "HYD-PUMP-737-11",
    "CAB-DOOR-ACT-321N-03", "WHL-NLG-737-22", "FMS-A321-NEO-09",
]
AIRCRAFT_TYPES = ["A330-300", "B777-300ER", "B737-800", "A321neo", "B787-9", "A350-900"]
AIRPORTS = ["ICN", "GMP", "NRT", "HND", "CDG", "JFK", "SIN", "HKG", "FRA", "LAX"]

STEP_NAMES = {
    1: "1단계 · FAK(Fly Away Kit) 재고 확인",
    2: "2단계 · Allocation 재고 확인",
    3: "3단계 · Pooling 파트너사 재고 확인",
    4: "4단계 · Main Station 타 항공사 대여 요청",
    5: "5단계 · 동일 기종 타 항공사 대여 요청",
    6: "6단계 · 본사 직원 Hand-carry 이동",
}
TOTAL_STEPS = 6

DEFAULT_EMAIL_BODY_TEMPLATE = (
    "안녕하세요,\n\n{airport} 공항에서 {aircraft_type} 기종 AOG가 발생하여 자재({part_number}) 부족으로 "
    "긴급 지원을 요청드립니다.\n해당 자재를 보유하고 계시다면 대여 가능 여부와 최소 소요 시간을 회신해 "
    "주시면 감사하겠습니다.\n\n감사합니다."
)


# ----------------------------------------------------------------------------
# 2. 영속화 (JSON 파일 기반) — 관리자가 데이터 관리 화면에서 수정하면 즉시 저장되어
#    다음 실행/세션에도 유지된다.
# ----------------------------------------------------------------------------

def load_db():
    if os.path.exists(DB_PATH):
        try:
            with open(DB_PATH, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            pass
    db = copy.deepcopy(DEFAULT_DB)
    save_db(db)
    return db


def save_db(db):
    with open(DB_PATH, "w", encoding="utf-8") as f:
        json.dump(db, f, ensure_ascii=False, indent=2)


if "db" not in st.session_state:
    st.session_state.db = load_db()
if "case" not in st.session_state:
    st.session_state.case = None


# ----------------------------------------------------------------------------
# 3. 조회 로직 — 사용자가 자유롭게 입력한 텍스트도 매칭되도록 대소문자/공백을 정규화한다.
# ----------------------------------------------------------------------------

def norm(s):
    return (s or "").strip().upper()


def tel_href(contact):
    return re.sub(r"[^0-9+]", "", contact or "")


def find_one(records, **filters):
    for r in records:
        if all(norm(r.get(k)) == norm(v) for k, v in filters.items()):
            return r
    return None


def evaluate_step(step, db, aircraft_type, part_number, airport):
    """단계별 재고/대여 확보 여부를 판단한다.
    반환값: (found: bool, message: str, detail: dict|None)
    detail에는 후속 액션 패널(전화/메일/편명 조회)에 필요한 원본 레코드를 담는다.
    """

    if step == 1:
        hit = find_one(db["fak_stock"], aircraft_type=aircraft_type, part_number=part_number)
        if hit:
            return True, (f"✅ **FAK 재고 확인됨**\n- 위치: {hit['location']} · 가용 수량: {hit['qty']}개\n\n"
                           f"당사 FAK 재고로 즉시 조치 가능합니다."), hit
        return False, "❌ FAK(Fly Away Kit)에는 해당 자재 재고가 없습니다.", None

    if step == 2:
        hit = find_one(db["allocation_stock"], aircraft_type=aircraft_type, part_number=part_number)
        if hit:
            return True, (f"✅ **Allocation 재고 확인됨**\n- 위치: {hit['location']} · 가용 수량: {hit['qty']}개\n\n"
                           f"당사 Allocation 재고로 조치 가능합니다."), hit
        return False, "❌ 당사 Allocation 재고에도 해당 자재가 없습니다.", None

    if step == 3:
        hit = find_one(db["pooling_stock"], aircraft_type=aircraft_type, part_number=part_number)
        if hit:
            partner_info = find_one(db["pooling_partners"], partner=hit["partner"]) or {}
            detail = {**hit, "contact": partner_info.get("contact", ""), "email": partner_info.get("email", "")}
            return True, (f"✅ **Pooling 파트너사 재고 확인됨 — {hit['partner']}**\n"
                           f"- 위치: {hit['location']} · 가용 수량: {hit['qty']}개\n"
                           f"- 연락처: {detail['contact']}\n\n"
                           f"Pooling 계약에 따라 대여 요청이 가능합니다."), detail
        checked = ", ".join(p["partner"] for p in db["pooling_partners"]) or "등록된 파트너사 없음"
        return False, f"❌ 조회한 Pooling 파트너사({checked}) 중 해당 자재 재고를 보유한 곳이 없습니다.", None

    if step == 4:
        queried = [r for r in db["main_station_airlines"] if norm(r["airport"]) == norm(airport)]
        hit = find_one(db["other_airline_station_stock"], airport=airport, part_number=part_number)
        detail = {"queried_airlines": queried, "hit": hit}
        if hit:
            return True, (f"✅ **Main Station 타 항공사 재고 확인됨 — {hit['airline']}**\n"
                           f"- 위치: {hit['location']} · 가용 수량: {hit['qty']}개\n- 연락처: {hit['contact']}\n\n"
                           f"{airport}을 Main Station으로 운영 중인 항공사로부터 대여 가능합니다."), detail
        op_names = ", ".join(o["airline"] for o in queried) if queried else "등록된 항공사 없음"
        return False, f"❌ {airport}을 Main Station으로 운영하는 항공사({op_names})에 문의했으나 재고가 없습니다.", detail

    if step == 5:
        # 타 항공사의 실제 대여 가능 재고는 사전에 알 수 없는 기밀 정보이므로,
        # "운영 항공사" 목록만으로 문의 대상을 좁히고 결과는 항상 "직접 문의 필요"로 처리한다.
        queried = [r for r in db["fleet_operators"] if norm(r["aircraft_type"]) == norm(aircraft_type)]
        detail = {"queried_airlines": queried}
        op_names = ", ".join(o["airline"] for o in queried) if queried else "등록된 항공사 없음"
        return False, (f"ℹ️ {aircraft_type} 기종을 운영 중인 항공사({op_names})의 실제 재고 보유 여부는 "
                        f"사전에 확인할 수 없습니다 (항공사 기밀 정보). 개별 문의가 필요합니다."), detail

    if step == 6:
        flights = fetch_flight_schedule(airport)[:3]
        detail = {"flights": flights, "customs": db["customs_team"]}
        return True, ("🧳 **본사 직원 Hand-carry 이동 확정**\n"
                       "모든 사전 단계에서 재고를 찾지 못해 최종 수단으로 조치합니다. "
                       "통관팀과 가장 빠른 편명을 확인하세요."), detail

    raise ValueError(f"알 수 없는 단계: {step}")


# ----------------------------------------------------------------------------
# 3.5. 실시간 항공편 / 공항 정박 데이터 공급자
#    지금은 더미 데이터를 생성해 반환하지만, 실제 항공편 API가 준비되면
#    _fetch_raw_flight_feed() / fetch_airport_parking_status() 내부의 "실 API 연동" 블록만
#    구현하면 된다 — 반환 스키마(딕셔너리 키)만 그대로 유지하면 맵/프로세스 코드는 수정할 필요가 없다.
#    Hand-carry 편명은 더 이상 "데이터 관리"에서 수동으로 입력하지 않는다: 실무에서도 편명/스케줄은
#    수시로 바뀌므로 대한항공 편명 API를 계속 조회해 최신 상태를 반영한다고 가정한다.
# ----------------------------------------------------------------------------

FLIGHT_API_CONFIG = {
    "enabled": False,  # 실 API가 준비되면 True로 바꾸고 base_url/api_key를 채운다
    "base_url": "https://api.example-airline.com/v1/flight-schedule",
    "api_key": "",
}
PARKING_API_CONFIG = {
    "enabled": False,
    "base_url": "https://api.example-airline.com/v1/airport-stand-status",
    "api_key": "",
}

# (위도, 경도, 한글 표기) — 맵 표시용 고정 참조 데이터라 API 대상이 아니다.
AIRPORT_COORDS = {
    "ICN": (37.4602, 126.4407, "인천"),
    "GMP": (37.5583, 126.7906, "김포"),
    "NRT": (35.7647, 140.3864, "나리타"),
    "HND": (35.5494, 139.7798, "하네다"),
    "CDG": (49.0097, 2.5479, "파리 샤를드골"),
    "JFK": (40.6413, -73.7781, "뉴욕 JFK"),
    "SIN": (1.3644, 103.9915, "싱가포르 창이"),
    "HKG": (22.3080, 113.9185, "홍콩"),
    "FRA": (50.0379, 8.5622, "프랑크푸르트"),
    "LAX": (33.9416, -118.4085, "LA"),
}

# 더미 항공편 공급자가 참고하는 기준 스케줄(출발까지 남은/지난 시간, 총 비행시간).
# 실 API 연동 시에는 이 상수 대신 API 응답을 그대로 사용하면 된다.
_BASE_FLIGHT_TEMPLATE = [
    {"flight_no": "KE901", "airline": "대한항공", "origin": "ICN", "destination_airport": "CDG", "dep_offset_hours": -2.0, "duration_hours": 12.5},
    {"flight_no": "KE017", "airline": "대한항공", "origin": "ICN", "destination_airport": "CDG", "dep_offset_hours": 6.0, "duration_hours": 12.5},
    {"flight_no": "KE703", "airline": "대한항공", "origin": "GMP", "destination_airport": "HND", "dep_offset_hours": 1.0, "duration_hours": 2.6},
    {"flight_no": "KE705", "airline": "대한항공", "origin": "ICN", "destination_airport": "NRT", "dep_offset_hours": -0.5, "duration_hours": 2.5},
    {"flight_no": "KE723", "airline": "대한항공", "origin": "ICN", "destination_airport": "NRT", "dep_offset_hours": 5.0, "duration_hours": 2.5},
    {"flight_no": "KE081", "airline": "대한항공", "origin": "ICN", "destination_airport": "JFK", "dep_offset_hours": 8.0, "duration_hours": 14.0},
    {"flight_no": "KE082", "airline": "대한항공", "origin": "JFK", "destination_airport": "ICN", "dep_offset_hours": -3.0, "duration_hours": 14.0},
    {"flight_no": "KE643", "airline": "대한항공", "origin": "ICN", "destination_airport": "SIN", "dep_offset_hours": 3.0, "duration_hours": 6.5},
    {"flight_no": "KE603", "airline": "대한항공", "origin": "ICN", "destination_airport": "HKG", "dep_offset_hours": -1.0, "duration_hours": 3.7},
    {"flight_no": "KE931", "airline": "대한항공", "origin": "ICN", "destination_airport": "FRA", "dep_offset_hours": 9.0, "duration_hours": 11.7},
    {"flight_no": "KE011", "airline": "대한항공", "origin": "ICN", "destination_airport": "LAX", "dep_offset_hours": 4.0, "duration_hours": 11.2},
    {"flight_no": "KE012", "airline": "대한항공", "origin": "LAX", "destination_airport": "ICN", "dep_offset_hours": -5.0, "duration_hours": 11.2},
]

# 공항별 전체 스탠드 수 — 인프라 정보라 자주 바뀌지 않으므로 고정값, 사용 중(occupied) 수만 매 분 갱신되는 것으로 흉내낸다.
_PARKING_STAND_CAPACITY = {
    "ICN": 46, "GMP": 20, "NRT": 30, "HND": 28, "CDG": 60,
    "JFK": 55, "SIN": 35, "HKG": 40, "FRA": 58, "LAX": 50,
}


def _generate_dummy_live_flights():
    now = datetime.now()
    flights = []
    for t in _BASE_FLIGHT_TEMPLATE:
        dep_time = now + timedelta(hours=t["dep_offset_hours"])
        arr_time = dep_time + timedelta(hours=t["duration_hours"])
        if now <= dep_time:
            status, progress = "예정", 0.0
        elif now >= arr_time:
            status, progress = "도착", 1.0
        else:
            status = "비행 중"
            progress = (now - dep_time).total_seconds() / (arr_time - dep_time).total_seconds()
        flights.append({
            **t,
            "dep_time": dep_time,
            "arr_time": arr_time,
            "status": status,
            "progress": round(progress, 3),
            "hours_from_now": round(max((arr_time - now).total_seconds() / 3600, 0.0), 1),
        })
    return flights


@st.cache_data(ttl=60, show_spinner=False)
def _fetch_raw_flight_feed():
    if FLIGHT_API_CONFIG["enabled"]:
        # ---- 실 API 연동 지점 -------------------------------------------------
        # import requests
        # resp = requests.get(
        #     FLIGHT_API_CONFIG["base_url"],
        #     headers={"Authorization": f"Bearer {FLIGHT_API_CONFIG['api_key']}"},
        #     timeout=5,
        # )
        # resp.raise_for_status()
        # return resp.json()["flights"]  # 아래와 동일한 키(flight_no/airline/origin/
        #                                # destination_airport/status/progress/hours_from_now/
        #                                # dep_time/arr_time)로 매핑해서 반환하면 된다.
        raise NotImplementedError("FLIGHT_API_CONFIG['enabled']=True로 설정했다면 실제 API 연동 로직을 구현하세요.")
    return _generate_dummy_live_flights()


def fetch_live_flights():
    """맵/현황판용 — 예정·비행 중·도착 모든 상태의 항공편을 반환한다."""
    return _fetch_raw_flight_feed()


def fetch_flight_schedule(destination_airport=None):
    """Hand-carry 후보용 — 아직 출발하지 않은(예정) 편만, 도착까지 남은 시간이 짧은 순으로 반환한다."""
    flights = [f for f in fetch_live_flights() if f["status"] == "예정"]
    if destination_airport:
        flights = [f for f in flights if norm(f["destination_airport"]) == norm(destination_airport)]
    return sorted(flights, key=lambda f: f["hours_from_now"])


@st.cache_data(ttl=60, show_spinner=False)
def fetch_airport_parking_status():
    if PARKING_API_CONFIG["enabled"]:
        # ---- 실 API 연동 지점 -------------------------------------------------
        # import requests
        # resp = requests.get(PARKING_API_CONFIG["base_url"],
        #                      headers={"Authorization": f"Bearer {PARKING_API_CONFIG['api_key']}"}, timeout=5)
        # resp.raise_for_status()
        # return resp.json()["airports"]  # {공항코드: {"total_stands":.., "occupied":..}}
        raise NotImplementedError("PARKING_API_CONFIG['enabled']=True로 설정했다면 실제 API 연동 로직을 구현하세요.")

    rng = random.Random(datetime.now().strftime("%Y%m%d%H%M"))  # 1분 단위로만 값이 바뀌는 "실시간 느낌" 스냅샷
    return {
        ap: {"total_stands": total, "occupied": rng.randint(int(total * 0.4), int(total * 0.95))}
        for ap, total in _PARKING_STAND_CAPACITY.items()
    }


# ----------------------------------------------------------------------------
# 4. AI 요약 — 규칙 기반(항상 즉시 동작) + 로컬 LLM(기본 자동 실행, 실패해도 앱이 죽지 않음)
#    두 기능 모두 "데이터 관리" 화면의 최신 데이터로부터 파생된 case/detail을 근거로 삼는다.
# ----------------------------------------------------------------------------

def build_rule_based_briefing(case):
    lines = [
        "### 📌 케이스 개요",
        f"- 기종: **{case['aircraft_type']}**  ·  자재 넘버: **{case['part_number']}**  ·  공항: **{case['airport']}**",
        f"- 현재 단계: **{case['step']}/{TOTAL_STEPS} — {STEP_NAMES[case['step']]}**",
        "",
        "### 🔍 지금까지 조회 결과",
    ]
    for rec in case["records"]:
        icon = "✅" if rec["found"] else "❌"
        lines.append(f"- {icon} {STEP_NAMES[rec['step']]}")

    lines.append("")
    lines.append("### 💡 추천 행동")

    last = case["records"][-1]
    if case["finished"]:
        if last["found"]:
            action = {
                1: "요청 정비사에게 FAK 재고 확보 사실을 재연락하세요.",
                2: "해당 지점 Allocation 부서에 연락해 자재 반출을 요청하세요.",
                3: "Pooling 파트너사에 연락해 대여 절차를 진행하세요.",
                4: "Main Station 항공사와 대여 조건을 확정하세요.",
                6: "통관팀에 연락하고, 가장 빠른 편명으로 Hand-carry를 확정하세요.",
            }[last["step"]]
            lines.append(f"- 🏁 **{STEP_NAMES[last['step']]}**에서 자재가 확보되었습니다. {action}")
        elif last["step"] in (4, 5):
            lines.append(f"- 🏁 **{STEP_NAMES[last['step']]}**에서 담당자가 프로세스를 종료했습니다. 조회 대상 항공사 전체에 긴급 메일을 보내 응답을 기다리세요.")
        else:
            lines.append(f"- 🏁 담당자가 **{STEP_NAMES[last['step']]}** 단계에서 재고 미확보 상태로 프로세스를 수동 종료했습니다. 대체 조달 방안을 검토하세요.")
    else:
        if case["step"] >= TOTAL_STEPS:
            lines.append("- ▶ 이번 단계가 마지막 조달 경로입니다. Hand-carry 조치를 확정해주세요.")
        else:
            nxt = case["step"] + 1
            lines.append(f"- ▶ 현재 단계에서 재고를 찾지 못했다면 **{STEP_NAMES[nxt]}**로 진행하세요.")
        if case["step"] >= 4:
            lines.append("- ⚠️ 이미 4단계 이상 진행되어 지연 리스크가 높습니다. 병행해서 인접 항공사에 긴급 연락을 취하는 것을 권장합니다.")
    return "\n".join(lines)


@st.cache_resource(show_spinner=False)
def _load_local_llm():
    from transformers import pipeline
    return pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")


def generate_llm_briefing(context_text):
    """로컬 LLM으로 상황을 다듬어 본다. 어떤 이유로든 실패하면 None을 반환해
    호출부가 규칙 기반 요약만으로도 정상 동작하도록 한다 (AI는 어디까지나 보조 기능)."""
    try:
        pipe = _load_local_llm()
        messages = [
            {"role": "system", "content": "당신은 항공사 AOG 자재 수급을 돕는 간결한 한국어 어시스턴트입니다."},
            {"role": "user", "content": f"다음 상황을 2~3문장으로 브리핑하고, 담당자가 지금 해야 할 행동을 한 문장으로 제안해줘:\n{context_text}"},
        ]
        out = pipe(messages, max_new_tokens=180, do_sample=False)
        content = out[0]["generated_text"][-1]["content"]
        return content.strip() or None
    except Exception:
        return None


def generate_llm_email_body(case):
    """긴급 지원 요청 메일 '본문'을 로컬 LLM으로 작성한다. 실패 시 None (템플릿으로 대체)."""
    try:
        pipe = _load_local_llm()
        messages = [
            {"role": "system", "content": (
                "당신은 항공사 자재관리팀 소속으로, 타 항공사에 AOG 자재 지원을 요청하는 정중하고 간결한 "
                "한국어 비즈니스 이메일 '본문'만 작성하는 어시스턴트입니다. 인사말, 상황 설명, 요청 사항, "
                "맺음말을 포함해 4~6문장으로 작성하고 제목이나 다른 설명 없이 본문 텍스트만 출력하세요."
            )},
            {"role": "user", "content": (
                f"항공기 기종 {case['aircraft_type']}, 자재 넘버 {case['part_number']}, "
                f"AOG 발생 공항 {case['airport']}. 이 자재가 부족한 상황이니 보유하고 계시다면 "
                f"대여를 요청드린다는 내용으로 작성해줘."
            )},
        ]
        out = pipe(messages, max_new_tokens=220, do_sample=False)
        content = out[0]["generated_text"][-1]["content"]
        return content.strip() or None
    except Exception:
        return None


def use_llm_enabled():
    return HAS_TRANSFORMERS and st.session_state.get("use_local_llm", True)


def get_ai_briefing(case):
    """상황 브리핑을 반환한다 (텍스트, 출처["llm"|"rule"]). 케이스 상태가 바뀔 때만 재생성해
    불필요한 재추론을 피한다 — 이 함수는 버튼 없이 항상 자동으로 호출된다."""
    rule_text = build_rule_based_briefing(case)
    cache_key = (case["part_number"], case["aircraft_type"], case["airport"], case["step"], case["finished"], len(case["action_log"]))
    cache = st.session_state.get("ai_briefing_cache")
    if cache and cache["key"] == cache_key:
        return cache["text"], cache["source"]

    text, source = rule_text, "rule"
    if use_llm_enabled():
        try:
            with st.spinner("AI가 데이터 관리 화면의 최신 데이터를 바탕으로 상황을 분석하는 중..."):
                llm_text = generate_llm_briefing(rule_text)
        except Exception:
            llm_text = None  # generate_llm_briefing이 스스로 예외를 삼키지만, 만일을 대비한 2중 방어
        if llm_text:
            text, source = llm_text, "llm"

    st.session_state.ai_briefing_cache = {"key": cache_key, "text": text, "source": source}
    return text, source


def get_email_body(case, key_prefix):
    cache_key = (case["part_number"], case["aircraft_type"], case["airport"], key_prefix)
    cache = st.session_state.setdefault("email_body_cache", {})
    if cache.get("key") == cache_key:
        return cache["body"]

    body = None
    if use_llm_enabled():
        try:
            with st.spinner("AI가 요청 메일 초안을 작성하는 중..."):
                body = generate_llm_email_body(case)
        except Exception:
            body = None  # generate_llm_email_body가 스스로 예외를 삼키지만, 만일을 대비한 2중 방어
    if not body:
        body = DEFAULT_EMAIL_BODY_TEMPLATE.format(**case)

    st.session_state.email_body_cache = {"key": cache_key, "body": body}
    return body


# ----------------------------------------------------------------------------
# 5. 케이스(프로세스) 상태 관리
# ----------------------------------------------------------------------------

def start_case(aircraft_type, part_number, airport, mechanic_contact):
    case = {
        "aircraft_type": aircraft_type.strip(),
        "part_number": part_number.strip(),
        "airport": airport.strip(),
        "mechanic_contact": mechanic_contact.strip(),
        "step": 1,
        "finished": False,
        "records": [],
        "action_log": [],
        "started_at": datetime.now().strftime("%Y-%m-%d %H:%M"),
    }
    found, message, detail = evaluate_step(1, st.session_state.db, aircraft_type, part_number, airport)
    case["records"].append({"step": 1, "found": found, "message": message, "detail": detail})
    st.session_state.case = case
    st.session_state.ai_briefing_cache = None
    st.session_state.email_body_cache = {}


def proceed_step():
    case = st.session_state.case
    if case["finished"] or case["step"] >= TOTAL_STEPS:
        return
    nxt = case["step"] + 1
    found, message, detail = evaluate_step(nxt, st.session_state.db, case["aircraft_type"], case["part_number"], case["airport"])
    case["step"] = nxt
    case["records"].append({"step": nxt, "found": found, "message": message, "detail": detail})


def resolve_here():
    case = st.session_state.case
    if case["finished"]:
        return
    case["finished"] = True


# ----------------------------------------------------------------------------
# 6. 대시보드 화면 (메인) — 좌: AI 어시스턴트 / 우: 프로세스 진행 + 액션 패널
# ----------------------------------------------------------------------------

def render_case_intake():
    with st.form("case_form", clear_on_submit=False):
        c1, c2, c3 = st.columns(3)
        aircraft_type = c1.text_input("항공기 기종 (Aircraft Type)", placeholder="예: A330-300")
        part_number = c2.text_input("자재 넘버 (Part Number)", placeholder="예: OXY-GEN-A330-15")
        airport = c3.text_input("AOG 발생 공항 (Station)", placeholder="예: ICN")
        mechanic_contact = st.text_input("요청 정비사 연락처 (선택 — FAK 확보 시 재연락용)", placeholder="예: 010-1234-5678")
        submitted = st.form_submit_button("🚨 AOG 프로세스 시작", type="primary", use_container_width=True)

    if submitted:
        if not aircraft_type.strip() or not part_number.strip() or not airport.strip():
            st.warning("항공기 기종 / 자재 넘버 / 공항을 모두 입력해주세요.")
        else:
            start_case(aircraft_type, part_number, airport, mechanic_contact)
            st.rerun()


def render_ai_panel():
    st.markdown("## 🤖 AI 상황 요약 & 추천 행동")
    case = st.session_state.case
    if not case:
        st.info("위 입력창에서 케이스를 시작하면 이곳에 AI 브리핑이 자동으로 표시됩니다.")
        return

    with st.container(border=True):
        text, source = get_ai_briefing(case)
        st.markdown(text)

    if not HAS_TRANSFORMERS:
        st.caption("ℹ️ `transformers`가 설치되어 있지 않아 규칙 기반 요약으로 표시 중입니다.")
    elif not st.session_state.get("use_local_llm", True):
        st.caption("ℹ️ 사이드바에서 로컬 LLM이 꺼져 있어 규칙 기반 요약으로 표시 중입니다.")
    elif source == "rule":
        st.caption("ℹ️ 로컬 LLM 응답을 가져오지 못해 규칙 기반 요약으로 표시 중입니다 (네트워크 차단 시 정상적인 현상).")
    else:
        st.caption("✨ 로컬 LLM(Qwen2.5-0.5B)이 '데이터 관리' 화면의 최신 데이터를 근거로 다듬은 브리핑입니다.")


def render_process_panel():
    st.markdown("## 📋 프로세스 진행")
    case = st.session_state.case
    if not case:
        st.info("케이스가 시작되지 않았습니다.")
        return

    last = case["records"][-1]
    with st.container(border=True):
        st.markdown(f"#### 진행 단계 {case['step']}/{TOTAL_STEPS} — {STEP_NAMES[case['step']]}")
        if last["found"]:
            st.success(last["message"])
        else:
            st.warning(last["message"])

        is_last_step = case["step"] >= TOTAL_STEPS
        bc1, bc2 = st.columns(2)
        if not case["finished"] and not is_last_step:
            if bc1.button("▶ 다음 단계 진행", use_container_width=True, key="proceed_btn"):
                proceed_step()
                st.rerun()
        if not case["finished"]:
            label = "✅ Hand-carry 조치 확정 (프로세스 종료)" if is_last_step else "✅ 현재 단계에서 조치 (프로세스 종료)"
            if bc2.button(label, use_container_width=True, key="resolve_btn", type="primary"):
                resolve_here()
                st.rerun()

    if case["finished"]:
        render_action_panel(case)

    with st.expander("📜 처리 이력 (Audit Log)", expanded=False):
        for rec in case["records"]:
            icon = "✅" if rec["found"] else "❌"
            st.markdown(f"**{icon} {STEP_NAMES[rec['step']]}**\n\n{rec['message']}")
            st.divider()
        for entry in case["action_log"]:
            st.markdown(f"🗂️ {entry}")


def render_bulk_email_action(case, airlines, key_prefix):
    """조회 대상 항공사 전체에 보낼 긴급 지원 요청 메일을 작성한다.
    본문은 로컬 LLM이 작성하고, 실패 시 정해진 템플릿으로 대체된다."""
    recipients = [a["email"] for a in airlines if a.get("email")]
    subject = f"[긴급] AOG 자재 지원 요청 - {case['aircraft_type']} / {case['part_number']} / {case['airport']}"
    body = get_email_body(case, key_prefix)

    st.write(f"조회 대상 항공사: {', '.join(a['airline'] for a in airlines) or '없음'}")
    if recipients:
        mailto = "mailto:" + ",".join(recipients) + "?subject=" + urllib.parse.quote(subject) + "&body=" + urllib.parse.quote(body)
        st.markdown(f'<a href="{mailto}" target="_blank">✉️ 조회 대상 항공사 전체({len(recipients)}곳)에 긴급 메일 작성하기</a>', unsafe_allow_html=True)
    else:
        st.warning("등록된 이메일 주소가 없습니다. '데이터 관리'에서 항공사 이메일을 추가해주세요.")
    with st.expander("메일 내용 미리보기"):
        st.text(f"제목: {subject}\n\n{body}")
    if st.button("✉️ 긴급 메일 발송 완료로 기록", key=f"log_email_{key_prefix}"):
        case["action_log"].append(f"[{datetime.now().strftime('%H:%M')}] {len(recipients)}개 항공사에 긴급 메일 발송 완료")
        st.rerun()


def render_action_panel(case):
    st.markdown("#### 🎯 다음 조치")
    last = case["records"][-1]
    step, found, detail = last["step"], last["found"], last["detail"]

    if step in (1, 2, 3):
        if not found:
            st.info("이 단계에서는 재고를 찾지 못한 채 담당자가 프로세스를 수동 종료했습니다. 대체 조달 방안을 별도로 검토해주세요.")
            return

        if step == 1:
            st.write(f"📍 위치: {detail['location']} · 수량: {detail['qty']}개")
            if case["mechanic_contact"]:
                href = tel_href(case["mechanic_contact"])
                st.markdown(f'<a href="tel:{href}">📞 요청 정비사({case["mechanic_contact"]})에게 전화하기</a>', unsafe_allow_html=True)
            else:
                st.caption("정비사 연락처가 입력되지 않았습니다. 위 정보로 직접 재연락해주세요.")
            if st.button("☎️ 재연락 완료로 기록", key="log_step1"):
                case["action_log"].append(f"[{datetime.now().strftime('%H:%M')}] 정비사에게 FAK 재고 확보 사실 재연락 완료")
                st.rerun()

        elif step == 2:
            dept = find_one(st.session_state.db["allocation_dept_contacts"], airport=case["airport"])
            st.write(f"📍 위치: {detail['location']} · 수량: {detail['qty']}개")
            if dept:
                st.write(f"담당 부서: **{dept['department']}**")
                colA, colB = st.columns(2)
                colA.markdown(f'<a href="tel:{tel_href(dept["contact"])}">📞 {dept["contact"]}</a>', unsafe_allow_html=True)
                colB.markdown(f'<a href="mailto:{dept["email"]}">✉️ {dept["email"]}</a>', unsafe_allow_html=True)
            else:
                st.warning(f"{case['airport']} 지점의 Allocation 부서 연락처가 등록되어 있지 않습니다. '데이터 관리'에서 추가해주세요.")
            if st.button("☎️ Allocation 부서 연락 완료로 기록", key="log_step2"):
                case["action_log"].append(f"[{datetime.now().strftime('%H:%M')}] {case['airport']} Allocation 부서에 연락 완료")
                st.rerun()

        elif step == 3:
            st.write(f"📍 위치: {detail['location']} · 수량: {detail['qty']}개")
            colA, colB = st.columns(2)
            colA.markdown(f'<a href="tel:{tel_href(detail["contact"])}">📞 {detail["contact"]}</a>', unsafe_allow_html=True)
            if detail.get("email"):
                colB.markdown(f'<a href="mailto:{detail["email"]}">✉️ {detail["email"]}</a>', unsafe_allow_html=True)
            if st.button("☎️ Pooling 파트너사 연락 완료로 기록", key="log_step3"):
                case["action_log"].append(f"[{datetime.now().strftime('%H:%M')}] {detail['partner']}에 대여 연락 완료")
                st.rerun()

    elif step == 4:
        if found:
            st.write(f"📍 위치: {detail['hit']['location']} · 수량: {detail['hit']['qty']}개")
            colA, colB = st.columns(2)
            colA.markdown(f'<a href="tel:{tel_href(detail["hit"]["contact"])}">📞 {detail["hit"]["contact"]}</a>', unsafe_allow_html=True)
            colB.markdown(f'<a href="mailto:{detail["hit"]["email"]}">✉️ {detail["hit"]["email"]}</a>', unsafe_allow_html=True)
            if st.button("☎️ Main Station 항공사 연락 완료로 기록", key="log_step4_direct"):
                case["action_log"].append(f"[{datetime.now().strftime('%H:%M')}] {detail['hit']['airline']}에 대여 연락 완료")
                st.rerun()
            if st.checkbox("📧 혹시 몰라 다른 Main Station 항공사에도 공유 메일 보내기", key="show_step4_bulk_email"):
                render_bulk_email_action(case, detail["queried_airlines"], "step4")
        else:
            render_bulk_email_action(case, detail["queried_airlines"], "step4")

    elif step == 5:
        st.info("동일 기종 운영 항공사의 실제 재고 보유 여부는 사전에 알 수 없습니다 (항공사 기밀 정보). 아래로 문의 메일을 보내 확인하세요.")
        render_bulk_email_action(case, detail["queried_airlines"], "step5")

    elif step == 6:
        customs = detail["customs"]
        st.write(f"통관팀: **{customs['name']}** · {customs['contact']} · {customs['email']}")
        flights = detail["flights"]
        if flights:
            options = [
                f"{f['flight_no']} ({f['airline']}) — 약 {f['hours_from_now']}시간 후 도착 "
                f"[{(datetime.now() + timedelta(hours=f['hours_from_now'])).strftime('%m/%d %H:%M')} 예정]"
                for f in flights
            ]
            chosen = st.radio("Hand-carry로 활용할 편명을 선택하세요 (가장 빠른 편이 상단)", options, key="flight_choice")
            if st.button("✅ 이 편으로 Hand-carry 확정", key="log_step6"):
                case["action_log"].append(f"[{datetime.now().strftime('%H:%M')}] 통관팀 연계 완료, Hand-carry 편명 확정: {chosen}")
                st.rerun()
        else:
            st.warning(f"{case['airport']}행으로 예정된 편명을 실시간 항공편 조회에서 찾지 못했습니다. '🗺️ 실시간 운항 현황'에서 다른 경유편을 확인해주세요.")


def render_dashboard_page():
    st.title("🚨 AOG 자재 수급 어시스턴트")
    st.caption("항공기 기종 / 자재 넘버 / AOG 발생 공항을 직접 입력해 프로세스를 시작하세요.")

    render_case_intake()

    if st.session_state.case:
        st.divider()
        left, right = st.columns(2)
        with left:
            render_ai_panel()
        with right:
            render_process_panel()

        st.divider()
        if st.button("🔄 새 AOG 건 시작 (현재 케이스 초기화)"):
            st.session_state.case = None
            st.session_state.ai_briefing_cache = None
            st.session_state.email_body_cache = {}
            st.rerun()


# ----------------------------------------------------------------------------
# 7. 데이터 관리 화면 — 주기적으로 바뀌는 FAK/Allocation/Pooling/공항별 계약 등을
#    비개발자도 표 형태로 직접 수정할 수 있게 한다. 저장 시 JSON 파일에 즉시 반영된다.
# ----------------------------------------------------------------------------

def df_editor_save(key, columns, int_cols=()):
    db = st.session_state.db
    df = pd.DataFrame(db[key])
    if df.empty:
        df = pd.DataFrame(columns=columns)
    edited = st.data_editor(df, num_rows="dynamic", use_container_width=True, key=f"editor_{key}")
    if st.button("💾 저장", key=f"save_{key}"):
        cleaned = edited.fillna("")
        for c in int_cols:
            if c in cleaned.columns:
                cleaned[c] = pd.to_numeric(cleaned[c], errors="coerce").fillna(0).astype(int)
        db[key] = cleaned.to_dict("records")
        save_db(db)
        st.success("저장되었습니다.")


def render_admin_page():
    st.title("⚙️ 데이터 관리")
    st.caption("FAK/Allocation 재고, Pooling 계약, 공항·기종별 연락처 등은 주기적으로 바뀌므로 여기서 직접 갱신하세요. 저장 즉시 대시보드와 AI 브리핑에 반영됩니다.")

    if os.path.exists(DB_PATH):
        mtime = datetime.fromtimestamp(os.path.getmtime(DB_PATH)).strftime("%Y-%m-%d %H:%M")
        st.caption(f"마지막 저장: {mtime}")

    st.info("🛬 Hand-carry 편명 스케줄은 여기서 수동으로 관리하지 않습니다 — 실시간 항공편 API(현재는 더미 공급자)에서 "
            "자동으로 조회됩니다. 확인은 '🗺️ 실시간 운항 현황' 화면에서 하세요.")

    tabs = st.tabs([
        "🏭 FAK 재고", "📦 Allocation 재고", "🤝 Pooling 파트너/재고",
        "🛫 공항별 Main Station/대여재고", "✈️ 기종별 운영 항공사",
        "📇 Allocation 부서·통관팀",
    ])

    with tabs[0]:
        df_editor_save("fak_stock", ["aircraft_type", "part_number", "qty", "location"], int_cols=["qty"])

    with tabs[1]:
        df_editor_save("allocation_stock", ["aircraft_type", "part_number", "qty", "location"], int_cols=["qty"])

    with tabs[2]:
        st.markdown("**파트너사 목록**")
        df_editor_save("pooling_partners", ["partner", "contact", "email"])
        st.markdown("**파트너사 보유 재고**")
        df_editor_save("pooling_stock", ["partner", "aircraft_type", "part_number", "qty", "location"], int_cols=["qty"])

    with tabs[3]:
        st.markdown("**공항별 Main Station 운영 항공사**")
        df_editor_save("main_station_airlines", ["airport", "airline", "contact", "email"])
        st.markdown("**공항별 대여 가능 재고 (4단계 매칭용)**")
        df_editor_save("other_airline_station_stock",
                        ["airport", "part_number", "airline", "qty", "location", "contact", "email"], int_cols=["qty"])

    with tabs[4]:
        st.caption("타 항공사의 실제 대여 가능 재고는 항공사 기밀이라 사전 등록하지 않습니다 — 이 목록은 5단계에서 문의 대상을 좁히는 용도로만 쓰입니다.")
        st.markdown("**기종별 운영 항공사**")
        df_editor_save("fleet_operators", ["aircraft_type", "airline", "contact", "email"])

    with tabs[5]:
        st.markdown("**공항별 Allocation 부서 연락처**")
        df_editor_save("allocation_dept_contacts", ["airport", "department", "contact", "email"])
        st.markdown("**통관팀 연락처**")
        db = st.session_state.db
        c1, c2, c3 = st.columns(3)
        name = c1.text_input("담당팀명", value=db["customs_team"]["name"], key="customs_name")
        contact = c2.text_input("연락처", value=db["customs_team"]["contact"], key="customs_contact")
        email = c3.text_input("이메일", value=db["customs_team"]["email"], key="customs_email")
        if st.button("💾 저장", key="save_customs"):
            db["customs_team"] = {"name": name, "contact": contact, "email": email}
            save_db(db)
            st.success("저장되었습니다.")


# ----------------------------------------------------------------------------
# 8. 실시간 운항 현황 화면 — 지도에 항공편 위치 + 공항 정박(Stand) 혼잡도를 표시한다.
#    데이터는 위 "3.5" 섹션의 더미 공급자에서 가져오며, 실 API 연동 시 이 화면은 코드 변경 없이
#    그대로 최신 데이터를 반영한다.
# ----------------------------------------------------------------------------

def build_flight_map(flights, parking):
    fig = go.Figure()

    lats, lons, texts, colors = [], [], [], []
    for code, (lat, lon, name) in AIRPORT_COORDS.items():
        info = parking.get(code, {"total_stands": 0, "occupied": 0})
        rate = info["occupied"] / info["total_stands"] if info["total_stands"] else 0
        lats.append(lat)
        lons.append(lon)
        texts.append(f"{name} ({code})<br>정박 {info['occupied']}/{info['total_stands']} ({rate:.0%})")
        colors.append(rate)

    fig.add_trace(go.Scattergeo(
        lat=lats, lon=lons, text=texts, mode="markers",
        marker=dict(
            size=16, color=colors,
            colorscale=[[0, "#0ca30c"], [0.7, "#fab219"], [1, "#d03b3b"]],
            cmin=0, cmax=1, showscale=True, colorbar=dict(title="정박 혼잡도", tickformat=".0%"),
            line=dict(width=1, color="#ffffff"),
        ),
        name="공항", hoverinfo="text",
    ))

    for f in flights:
        if f["origin"] not in AIRPORT_COORDS or f["destination_airport"] not in AIRPORT_COORDS:
            continue
        olat, olon, _ = AIRPORT_COORDS[f["origin"]]
        dlat, dlon, _ = AIRPORT_COORDS[f["destination_airport"]]
        fig.add_trace(go.Scattergeo(
            lat=[olat, dlat], lon=[olon, dlon], mode="lines",
            line=dict(width=1, color="#2a78d6"), opacity=0.35,
            showlegend=False, hoverinfo="skip",
        ))
        if f["status"] == "비행 중":
            plat = olat + (dlat - olat) * f["progress"]
            plon = olon + (dlon - olon) * f["progress"]
            fig.add_trace(go.Scattergeo(
                lat=[plat], lon=[plon], mode="markers",
                marker=dict(size=11, symbol="triangle-up", color="#eb6834", line=dict(width=1, color="#ffffff")),
                text=f"{f['flight_no']} ({f['airline']})<br>{f['origin']} → {f['destination_airport']}<br>진행률 {f['progress']:.0%}",
                hoverinfo="text", showlegend=False,
            ))

    fig.update_geos(projection_type="natural earth", showland=True, landcolor="#f0efec",
                     showcountries=True, countrycolor="#c3c2b7", showocean=True, oceancolor="#fcfcfb")
    fig.update_layout(height=550, margin=dict(l=0, r=0, t=10, b=0),
                       font=dict(family="system-ui, -apple-system, 'Segoe UI', sans-serif"))
    return fig


def render_map_page():
    st.title("🗺️ 실시간 운항 & 공항 정박 현황")
    st.caption("항공편 위치·정박 현황은 더미 데이터로 시뮬레이션됩니다. 실제 API가 연동되면 코드 변경 없이 이 화면이 그대로 실시간 데이터를 반영하도록 설계되어 있습니다.")

    if st.button("🔄 새로고침"):
        _fetch_raw_flight_feed.clear()
        fetch_airport_parking_status.clear()
        st.rerun()

    flights = fetch_live_flights()
    parking = fetch_airport_parking_status()

    st.plotly_chart(build_flight_map(flights, parking), use_container_width=True)

    col1, col2 = st.columns(2)
    with col1:
        st.markdown("#### ✈️ 실시간 항공편")
        df = pd.DataFrame(flights)
        df["출발"] = df["dep_time"].dt.strftime("%m/%d %H:%M")
        df["도착 예정"] = df["arr_time"].dt.strftime("%m/%d %H:%M")
        show = df[["flight_no", "airline", "origin", "destination_airport", "status", "출발", "도착 예정"]]
        show.columns = ["편명", "항공사", "출발지", "도착지", "상태", "출발", "도착 예정"]
        st.dataframe(show, use_container_width=True, hide_index=True)
    with col2:
        st.markdown("#### 🛬 공항별 정박(Stand) 현황")
        rows = [
            {"공항": f"{AIRPORT_COORDS.get(ap, (0, 0, ap))[2]} ({ap})", "전체 스탠드": v["total_stands"],
             "사용 중": v["occupied"], "혼잡도": f"{v['occupied'] / v['total_stands']:.0%}"}
            for ap, v in parking.items()
        ]
        st.dataframe(pd.DataFrame(rows), use_container_width=True, hide_index=True)


# ----------------------------------------------------------------------------
# 9. 사이드바 내비게이션 — 대시보드 / 실시간 운항 현황 / 데이터 관리 세 화면 전환
# ----------------------------------------------------------------------------

with st.sidebar:
    st.markdown("## ✈️ AOG 어시스턴트")
    page = st.radio("화면 선택", ["🚨 AOG 대시보드", "🗺️ 실시간 운항 현황", "⚙️ 데이터 관리"], key="nav_page")
    st.divider()
    if HAS_TRANSFORMERS:
        st.checkbox(
            "🧠 로컬 LLM 사용 (상황 요약 · 메일 초안)",
            value=True, key="use_local_llm",
            help="끄면 규칙 기반 요약과 기본 메일 템플릿이 즉시(지연 없이) 표시됩니다.",
        )
    else:
        st.caption("ℹ️ transformers 미설치 - 규칙 기반 요약만 사용")
    st.caption("Engine: Streamlit · 로컬 LLM(선택) · JSON 영속화 · 항공편 API 연동 포인트 내장")

if page == "🚨 AOG 대시보드":
    render_dashboard_page()
elif page == "🗺️ 실시간 운항 현황":
    render_map_page()
else:
    render_admin_page()


## Cell 3 — 앱 실행 (Colab 자체 포트포워딩만 사용, 외부 터널 불필요)

npm/localtunnel 같은 제3자 서비스를 전혀 쓰지 않고, Google Colab이 공식 제공하는 커널 포트포워딩(`google.colab.output`)만 사용합니다. 이미 접속 중인 `colab.research.google.com` 도메인을 그대로 경유하므로 사내망 보안 정책에 걸릴 일이 없습니다.

In [ ]:
import subprocess, time

LOG_PATH = "/content/streamlit_log.txt"

# 기존에 떠 있던 프로세스가 있으면 정리
subprocess.run(["pkill", "-f", "streamlit run app.py"], stderr=subprocess.DEVNULL)

process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--server.enableWebsocketCompression", "false",
    ],
    stdout=open(LOG_PATH, "w"),
    stderr=subprocess.STDOUT,
)

booted = False
for _ in range(30):
    time.sleep(1)
    log_text = open(LOG_PATH).read()
    if "You can now view your Streamlit app" in log_text or "Local URL" in log_text:
        booted = True
        break

if not booted:
    print("Streamlit이 기동되지 않았습니다. 아래 로그를 확인하세요.\n")
    print(log_text)
else:
    from google.colab import output
    print("Streamlit 정상 기동. 아래에서 새 창으로 대시보드를 엽니다.")
    output.serve_kernel_port_as_window(8501)

    # 새 창 팝업이 막혀 있다면 아래 줄의 주석을 풀어 노트북 내부에 바로 임베드할 수 있습니다.
    # output.serve_kernel_port_as_iframe(8501, height=900)
